# 1. データの概要を確認する

タイタニック号の乗客名簿を使って、**どんな人が生き残ったのかを言い当てる**のがこの課題。

名簿は2つに分かれている。

- **学習用データ** — 助かったかどうかの答えが付いている。ここから法則を見つける
- **評価用データ** — 答えが隠されている。この人たちの生死を当てて提出する

まずは、どんな項目が並んでいるのか、値が入っていない場所がないかを見ていく。

## 1. データを読み込む

CSV ファイルを Python で扱える表の形にする。

表を扱う道具として **pandas** を使う。Excel のような表を、コードで操作できるようにする道具。

In [ ]:
# import は道具を呼び出す命令。as pd は「以降 pd と略して書く」という宣言
import pandas as pd

# read_csv で CSV ファイルを読み込む
#
# index_col=0 : 1 列目（id）を「行の名前」として扱う、という指定
#               名前にしておかないと id が普通のデータとして扱われ、
#               平均を計算するときなどに混ざってしまう
#               0 が 1 列目を指すのは、Python が 0 から数えるため
train = pd.read_csv("../../data/raw/train.csv", index_col=0)  # 学習用（答えあり）
test = pd.read_csv("../../data/raw/test.csv", index_col=0)  # 評価用（答えなし）

# header=None : このファイルは 1 行目から中身が始まる、という指定
#               付けないと 1 人目のデータが列の名前だと勘違いされて消える
sample_submit = pd.read_csv("../../data/raw/sample_submit.csv", index_col=0, header=None)  # 提出の見本

## 2. データの大きさを確認する

読み込んだ3つが、それぞれ何人分・何項目あるのかを見る。

In [ ]:
# .shape は表の大きさを (行数, 列数) の形で教えてくれる
# print() で囲んでいるのは、1 つのセルで 3 つ表示したいから
#   囲まないと最後の 1 つしか表示されない
print("train:", train.shape)  # (445, 8) → 445 人 × 8 項目
print("test:", test.shape)  # (446, 7) → 1 項目少ない
print("sample_submit:", sample_submit.shape)  # (446, 1) → 予測を書く列だけ

# train と test の差の 1 項目が survived（助かったかどうか）。これを当てるのが課題
# id が項目数に入っていないのは、index_col=0 で行の名前にしたため

## 3. 中身を先頭だけ見る

どんな項目があって、どんな値が入っているのかを実際に見てみる。
445 人すべてを表示すると画面が埋まるので、先頭の5人だけにする。

In [ ]:
# .head() は先頭 5 行だけ表示する
#   head(3) と書けば 3 行、tail() なら末尾 5 行
#   セルの最後に書いた式は自動で表示されるので、print() は付けない
train.head()

# 出力の見方
#   左端の太い数字が id。3, 4, 7... と飛んでいるのは、間の番号が test 側にあるため
#   NaN は「値が入っていない」印。この状態を欠損（けっそん）と呼ぶ
#   age が 35.0 と小数なのは、値が入っていない人がいる列を
#   pandas が自動で小数として扱うため。年齢が小数という意味ではない

In [ ]:
test.head()

## 4. 項目ごとの状態を確認する

ここがこの章の本題。項目ごとに次の2つを調べる。

- **値が入っていない人が何人いるか** — 入っていないと計算できない
- **数字なのか文字なのか** — 文字のままでは計算に使えない

ここで見つかった問題が、そのまま次の章でやる作業になる。

In [ ]:
# .info() は項目ごとの状態を一覧にする
#   .head() が「中身を数人分見る」のに対し、.info() は「項目全体の状態を見る」もの
#   表を返すのではなく、その場に直接書き出すので print() は付けない
train.info()

# 出力の見方
#   Non-Null Count : 値が入っている人数。445 より少なければ、その差だけ空いている
#   Dtype          : 入っている値の種類
#     int64   整数（例: pclass の 1, 2, 3）
#     float64 小数（例: fare の 53.1）
#     str     文字（例: sex の male / female）

In [ ]:
test.info()

### 分かったこと

2つの出力から、次の章で手を入れる必要がある箇所が3つ見つかる。

**1. `age`（年齢）が空いている人がいる**

train は 445 人中 360 人、test は 446 人中 354 人しか値がない。
空いている人の行をまるごと捨てる手もあるが、それだと学習に使えるデータが2割近く減ってしまう。
何かの値で埋めるか、年齢そのものを使わないかを決める必要がある。

**2. `embarked`（乗った港）が2人分空いている**

train のみで、test には無い。2人だけなので影響は小さいが、
空いたままでは計算に使えないので、やはり埋める必要がある。

**3. `sex` と `embarked` が文字で入っている**

この後で使う予測の仕組みは、**数字しか受け付けない**。
`male` / `female` のような文字は、数字に置き換えないと渡せない。

`pclass`（客室の等級）は 1・2・3 という数字だが、中身は「上等・中等・下等」という区分。
数えた量ではなく順位なので、**1 と 2 の差が「1 人分」のような意味は持たない**。

ただし順番そのものには意味があり、**数字が小さいほど良い客室**。
豪華さと数字の大小が逆向きなので、後の章で符号を読むときに混乱しやすい点に注意する。